# 01 - Análisis demográfico (EPH)

Panorama demográfico de la población a partir de la EPH (INDEC), usando los parquets
compilados por el notebook **00_preparacion_bases** (en Google Drive, `carga_EPH/processed`).

**Contenido:**
1. Pirámide de población (edad × sexo) del último trimestre.
2. Composición de los hogares (relación de parentesco, tamaño del hogar).
3. Distribución poblacional por región y aglomerado.
4. Evolución temporal (población, edad promedio, índice de masculinidad).

> Todas las estadísticas se calculan **ponderadas con `PONDERA`** (factor de expansión),
> para que representen a la población y no a la muestra. Ver `.claude/memoria_EPH.md`
> para el diccionario de variables.

**Variables usadas (base Personas):** `CH06` (edad), `CH04` (sexo: 1=Varón, 2=Mujer),
`CH03` (parentesco), `REGION`, `AGLOMERADO`, `IX_TOT` (miembros del hogar), `PONDERA`.

## 1. Setup (Colab)

Clona el repo, monta Drive y define la carpeta de los parquets compilados. Si todavía no
corriste el notebook 00, hacelo primero (genera los parquets en `carga_EPH/processed`).

In [ ]:
import sys, os, shutil, glob

REPO_URL = "https://github.com/santiagoriverti/analisis_EPH.git"
REPO_DIR = "/content/analisis_EPH"

if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull -q
else:
    !git clone -q {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
sys.path.insert(0, REPO_DIR)

from google.colab import drive
drive.mount('/content/drive')

# Los parquets compilados viven en Drive, pero leer muchos archivos por FUSE puede
# desconectar el mount ("Transport endpoint is not connected"). Por eso los copiamos
# UNA vez a disco local de Colab y leemos de ahí (rápido y estable).
DRIVE_PROCESSED = "/content/drive/MyDrive/carga_EPH/processed"
PROCESSED_DIR = "/content/processed_local"
os.makedirs(PROCESSED_DIR, exist_ok=True)

for src in glob.glob(os.path.join(DRIVE_PROCESSED, "eph_T*.parquet")):
    dst = os.path.join(PROCESSED_DIR, os.path.basename(src))
    if not os.path.exists(dst):
        shutil.copy(src, dst)

n = len(glob.glob(os.path.join(PROCESSED_DIR, "eph_T*.parquet")))
print(f"Parquets locales listos en {PROCESSED_DIR}: {n} trimestres")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_panel, list_available_quarters, _period_tag

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

# Trimestres disponibles (los que tienen .zip en carga_EPH y por ende parquet compilado)
quarters = list_available_quarters()
ULTIMO = quarters[-1]
print("Trimestres disponibles:", [_period_tag(y, p) for y, p in quarters])
print("Último trimestre:", _period_tag(*ULTIMO))

## 2. Pirámide de población (edad × sexo)

Población por grupos quinquenales de edad y sexo, en el último trimestre, expandida con
`PONDERA`. Varones a la izquierda (valores negativos), mujeres a la derecha.

In [ ]:
dem = load_panel(
    columns=["CH06", "CH04", "PONDERA"],
    quarters=[ULTIMO],
    out_dir=PROCESSED_DIR,
)

# Grupos quinquenales de edad (0-4, 5-9, ..., 80+)
bins = list(range(0, 85, 5)) + [200]
labels = [f"{b}-{b+4}" for b in range(0, 80, 5)] + ["80+"]
dem = dem[dem["CH06"].between(0, 120)].copy()
dem["grupo_edad"] = pd.cut(dem["CH06"], bins=bins, labels=labels, right=False)

pir = (
    dem.groupby(["grupo_edad", "CH04"], observed=True)["PONDERA"].sum().unstack("CH04")
    .rename(columns={1: "Varones", 2: "Mujeres"})
)
pir

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
y = np.arange(len(pir.index))
ax.barh(y, -pir["Varones"] / 1e3, color="#4C72B0", label="Varones")
ax.barh(y, pir["Mujeres"] / 1e3, color="#C44E52", label="Mujeres")
ax.set_yticks(y)
ax.set_yticklabels(pir.index)
ax.set_xlabel("Población (miles)")
ax.set_ylabel("Grupo de edad")
ax.set_title(f"Pirámide de población - T{ULTIMO[1]} {ULTIMO[0]} (EPH, ponderado)")
# Eje x en valores absolutos
ax.set_xticklabels([f"{abs(int(t))}" for t in ax.get_xticks()])
ax.legend()
plt.tight_layout()
plt.show()

## 3. Composición de los hogares

Distribución de la relación de parentesco (`CH03`) y tamaño de los hogares (`IX_TOT`).

In [ ]:
PARENTESCO = {
    1: "Jefe/a", 2: "Cónyuge/pareja", 3: "Hijo/a/hijastro/a", 4: "Yerno/nuera",
    5: "Nieto/a", 6: "Madre/padre", 7: "Suegro/a", 8: "Hermano/a",
    9: "Otros familiares", 10: "No familiares",
}

par = load_panel(columns=["CH03", "PONDERA"], quarters=[ULTIMO], out_dir=PROCESSED_DIR)
comp = par.groupby("CH03")["PONDERA"].sum()
comp.index = comp.index.map(lambda c: PARENTESCO.get(int(c), str(c)))
comp = (comp / comp.sum() * 100).sort_values(ascending=True)

ax = comp.plot(kind="barh", color="#55A868")
ax.set_xlabel("% de la población")
ax.set_title(f"Composición de hogares por parentesco - T{ULTIMO[1]} {ULTIMO[0]}")
for i, v in enumerate(comp):
    ax.text(v + 0.3, i, f"{v:.1f}%", va="center")
plt.tight_layout()
plt.show()

In [ ]:
# Tamaño del hogar: distribución de hogares según cantidad de miembros (IX_TOT).
# Tomamos un registro por hogar (el jefe, CH03==1) para no contar el hogar N veces.
hog = load_panel(
    columns=["CH03", "IX_TOT", "PONDERA"], quarters=[ULTIMO], out_dir=PROCESSED_DIR
)
jefes = hog[hog["CH03"] == 1].copy()
jefes["tam"] = jefes["IX_TOT"].clip(upper=7)  # 7 = "7 o más"
dist_tam = jefes.groupby("tam")["PONDERA"].sum()
dist_tam = (dist_tam / dist_tam.sum() * 100)
dist_tam.index = [f"{int(i)}" if i < 7 else "7+" for i in dist_tam.index]

ax = dist_tam.plot(kind="bar", color="#8172B3")
ax.set_xlabel("Cantidad de miembros del hogar")
ax.set_ylabel("% de hogares")
ax.set_title(f"Distribución del tamaño de los hogares - T{ULTIMO[1]} {ULTIMO[0]}")
promedio = np.average(jefes["IX_TOT"], weights=jefes["PONDERA"])
print(f"Tamaño promedio del hogar: {promedio:.2f} personas")
plt.tight_layout()
plt.show()

## 4. Distribución poblacional por región

Población expandida por región (`REGION`) en el último trimestre.

> Nota: la EPH es representativa de los **aglomerados urbanos** relevados, no del total país.
> Las cifras representan a la población de esos aglomerados.

In [ ]:
REGION = {1: "Gran Buenos Aires", 40: "Noroeste", 41: "Noreste",
          42: "Cuyo", 43: "Pampeana", 44: "Patagonia"}

reg = load_panel(columns=["REGION", "PONDERA"], quarters=[ULTIMO], out_dir=PROCESSED_DIR)
pob_reg = reg.groupby("REGION")["PONDERA"].sum().sort_values(ascending=True)
pob_reg.index = pob_reg.index.map(lambda r: REGION.get(int(r), str(r)))

ax = (pob_reg / 1e6).plot(kind="barh", color="#4C72B0")
ax.set_xlabel("Población (millones)")
ax.set_title(f"Población por región - T{ULTIMO[1]} {ULTIMO[0]} (aglomerados EPH)")
for i, v in enumerate(pob_reg / 1e6):
    ax.text(v + 0.05, i, f"{v:.2f}M", va="center")
plt.tight_layout()
plt.show()

## 5. Evolución temporal

Serie de toda la ventana disponible: edad promedio e índice de masculinidad
(varones por cada 100 mujeres), ponderados, por trimestre.

In [ ]:
serie = load_panel(columns=["CH06", "CH04", "PONDERA"], out_dir=PROCESSED_DIR)
serie = serie[serie["CH06"].between(0, 120)].copy()
serie["periodo"] = serie["ANIO"].astype(str) + "T" + serie["TRIMESTRE"].astype(str)

def _resumen(g):
    edad_prom = np.average(g["CH06"], weights=g["PONDERA"])
    varones = g.loc[g["CH04"] == 1, "PONDERA"].sum()
    mujeres = g.loc[g["CH04"] == 2, "PONDERA"].sum()
    return pd.Series({
        "edad_promedio": edad_prom,
        "indice_masculinidad": varones / mujeres * 100,
    })

evol = serie.groupby(["ANIO", "TRIMESTRE"]).apply(_resumen, include_groups=False)
evol.index = [f"{y}T{p}" for y, p in evol.index]
evol

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 9), sharex=True)

evol["edad_promedio"].plot(ax=ax1, marker="o", color="#4C72B0")
ax1.set_ylabel("Edad promedio (años)")
ax1.set_title("Edad promedio de la población (EPH, ponderado)")

evol["indice_masculinidad"].plot(ax=ax2, marker="o", color="#C44E52")
ax2.axhline(100, color="gray", ls="--", lw=1)
ax2.set_ylabel("Varones por 100 mujeres")
ax2.set_title("Índice de masculinidad")
ax2.set_xticks(range(len(evol.index)))
ax2.set_xticklabels(evol.index, rotation=90)

plt.tight_layout()
plt.show()

---
Notebook generado sobre los parquets compilados por `00_preparacion_bases`. Para agregar
trimestres nuevos, subir el `.zip` del INDEC a `carga_EPH` y re-correr el notebook 00.